# Custom Likelihoods in PyMC: One-Inflated Beta Regression for Loan Repayment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/one_inflated_beta_regression.ipynb)

This notebook implements a Bayesian One-Inflated Beta (OIB) regression model using PyMC. We model loan repayment fractions where data has a spike at 1.0 (fully repaid) and a continuous Beta distribution for partial repayments.

**Blog post:** [Custom Likelihoods in PyMC: One-Inflated Beta Regression](https://sesen.ai/blog/one-inflated-beta-regression-pymc)

**Prerequisites:** [Bayesian Survival Analysis with PyMC](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/bayesian_survival_analysis.ipynb)

In [ ]:
# Install dependencies (Colab)
# !pip install pymc arviz matplotlib numpy -q

In [ ]:
import numpy as np
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import matplotlib.pyplot as plt

print(f"PyMC version: {pm.__version__}")
print(f"ArviZ version: {az.__version__}")

## 1. Generate Synthetic Loan Data

We simulate 2,000 personal loans with four standardised covariates:
- **Credit score**: higher → more likely to fully repay
- **Loan-to-value ratio**: higher → less likely to fully repay
- **Interest rate**: higher → less likely to fully repay
- **Income ratio**: higher → more likely to fully repay

Each loan's repayment fraction is drawn from an OIB mixture:
- With probability π_i: fully repaid (y = 1.0)
- With probability 1 - π_i: partial repayment from Beta(θ_i·φ, (1-θ_i)·φ)

In [ ]:
np.random.seed(42)

N = 2000
credit_score = np.random.normal(0, 1, N)
loan_to_value = np.random.normal(0, 1, N)
interest_rate = np.random.normal(0, 1, N)
income_ratio = np.random.normal(0, 1, N)
X = np.column_stack([credit_score, loan_to_value, interest_rate, income_ratio])
feature_names = ['credit_score', 'loan_to_value', 'interest_rate', 'income_ratio']

# True parameters
true_psi = np.array([0.5, 0.8, -0.6, -0.4, 0.3])    # pi coefficients
true_delta = np.array([0.3, 0.5, -0.3, -0.2, 0.2])   # theta coefficients
true_phi = 5.0                                          # Beta precision

# Per-loan probability of full repayment (logistic link)
logit_pi = true_psi[0] + X @ true_psi[1:]
pi_true = 1 / (1 + np.exp(-logit_pi))

# Per-loan mean partial repayment (logistic link)
logit_theta = true_delta[0] + X @ true_delta[1:]
theta_true = 1 / (1 + np.exp(-logit_theta))

# Beta shape parameters from mean-precision
alpha_true = theta_true * true_phi
beta_true = (1 - theta_true) * true_phi

# Sample from the OIB mixture
u = np.random.uniform(size=N)
repayment = np.where(u < pi_true, 1.0, np.random.beta(alpha_true, beta_true))

n_full = (repayment == 1.0).sum()
print(f"Fully repaid: {n_full}/{N} ({n_full/N:.1%})")
print(f"Partial repayment: {N - n_full}/{N} ({(N - n_full)/N:.1%})")

### Visualise the Data

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
partial = repayment[repayment < 1.0]
ax.hist(partial, bins=40, density=False, alpha=0.7, color='steelblue',
        edgecolor='white', label='Partial repayment')
ax.bar(1.0, n_full, width=0.02, color='coral', alpha=0.8, edgecolor='white',
       label=f'Fully repaid (n={n_full})')
ax.set_xlabel('Repayment Fraction', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title('Loan Repayment Distribution: Spike at 1.0 + Beta Component', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(-0.02, 1.08)
plt.tight_layout()
plt.show()

## 2. Build the OIB Model

The OIB model is a mixture of a point mass at 1.0 and a Beta distribution.

We implement it using `pm.Potential` by splitting observations into two groups:
- **Fully repaid** (y = 1): contributes log(π_i) to the log-likelihood
- **Partial** (0 < y < 1): contributes log(1 - π_i) + log Beta(y_i | α_i, β_i)

This approach avoids numerical issues with `pt.switch` and boundary values.

In [ ]:
# Split observations by type
full_idx = np.where(repayment == 1.0)[0]
partial_idx = np.where(repayment < 1.0)[0]
partial_values = repayment[partial_idx]

with pm.Model() as oib_model:
    # Pi sub-model: probability of full repayment (logistic link)
    psi_intercept = pm.Normal('psi_intercept', mu=0, sigma=5)
    psi_coeffs = pm.Normal('psi_coeffs', mu=0, sigma=1, shape=4)
    logit_pi = psi_intercept + pt.dot(X, psi_coeffs)
    pi = pm.Deterministic('pi', pm.math.invlogit(logit_pi))

    # Theta sub-model: mean of partial repayment Beta (logistic link)
    delta_intercept = pm.Normal('delta_intercept', mu=0, sigma=5)
    delta_coeffs = pm.Normal('delta_coeffs', mu=0, sigma=1, shape=4)
    logit_theta = delta_intercept + pt.dot(X, delta_coeffs)
    theta = pm.Deterministic('theta', pm.math.invlogit(logit_theta))

    # Phi: Beta precision (shared across all loans)
    phi = pm.Gamma('phi', alpha=2, beta=0.5)

    # Convert mean-precision to standard Beta parameters
    a = theta * phi
    b = (1 - theta) * phi

    # Expected repayment: E[Y] = pi + (1 - pi) * theta
    E_f = pm.Deterministic('E_f', pi + (1 - pi) * theta)

    # --- Piecewise log-likelihood via pm.Potential ---
    # Fully repaid loans: log(pi_i)
    pm.Potential('ll_full', pt.sum(pt.log(pi[full_idx])))

    # Partial repayments: log(1 - pi_i) + log Beta(y_i | a_i, b_i)
    pa, pb = a[partial_idx], b[partial_idx]
    beta_logp = (pt.gammaln(pa + pb) - pt.gammaln(pa) - pt.gammaln(pb)
                 + (pa - 1) * pt.log(partial_values)
                 + (pb - 1) * pt.log(1 - partial_values))
    pm.Potential('ll_partial', pt.sum(pt.log(1 - pi[partial_idx]) + beta_logp))

print(oib_model.point_logps())

In [ ]:
with oib_model:
    trace = pm.sample(draws=1000, tune=3000, chains=4,
                      target_accept=0.95, random_seed=42,
                      init='jitter+adapt_diag')

print(az.summary(trace, var_names=['psi_intercept', 'psi_coeffs',
                                    'delta_intercept', 'delta_coeffs', 'phi']))

### MCMC Diagnostics

Check convergence: chains should mix well, R-hat < 1.01, ESS > 400.

In [ ]:
az.plot_trace(trace, var_names=['psi_intercept', 'psi_coeffs',
                                 'delta_intercept', 'delta_coeffs', 'phi'],
              compact=True, figsize=(14, 12))
plt.suptitle('OIB Model: Trace Plots', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

n_div = trace.sample_stats['diverging'].sum().values
print(f"Total divergences: {n_div}")

## 3. Parameter Recovery

Since we generated the data with known true parameters, we can verify that the model recovers them.

In [ ]:
summary = az.summary(trace, var_names=['psi_intercept', 'psi_coeffs',
                                        'delta_intercept', 'delta_coeffs', 'phi'])

true_vals = np.concatenate([true_psi, true_delta, [true_phi]])
param_names = (['psi_intercept'] + [f'psi_coeffs[{i}]' for i in range(4)] +
               ['delta_intercept'] + [f'delta_coeffs[{i}]' for i in range(4)] + ['phi'])

print("\nTrue vs Estimated:")
for name, true_val in zip(param_names, true_vals):
    est = summary.loc[name, 'mean']
    low = summary.loc[name, 'hdi_3%']
    high = summary.loc[name, 'hdi_97%']
    in_hdi = '\u2713' if low <= true_val <= high else '\u2717'
    print(f"  {name:20s}: true={true_val:6.2f}, est={est:6.3f} [{low:.3f}, {high:.3f}] {in_hdi}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
y_pos = np.arange(len(param_names))
means = summary['mean'].values
hdi_low = summary['hdi_3%'].values
hdi_high = summary['hdi_97%'].values

ax.errorbar(means, y_pos, xerr=[means - hdi_low, hdi_high - means],
            fmt='o', color='steelblue', capsize=4, label='Posterior (94% HDI)')
ax.scatter(true_vals, y_pos, marker='x', color='crimson', s=80,
           zorder=5, label='True value')
ax.set_yticks(y_pos)
ax.set_yticklabels(param_names, fontsize=10)
ax.axvline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Value', fontsize=12)
ax.legend(loc='lower right', fontsize=11)
ax.set_title('Parameter Recovery: Posterior vs True Values', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Posterior Predictive Check

Since we used `pm.Potential` rather than an observed distribution, we generate predictive samples manually from the posterior.

In [ ]:
psi_int_post = trace.posterior['psi_intercept'].values.flatten()
psi_coeff_post = trace.posterior['psi_coeffs'].values.reshape(-1, 4)
delta_int_post = trace.posterior['delta_intercept'].values.flatten()
delta_coeff_post = trace.posterior['delta_coeffs'].values.reshape(-1, 4)
phi_post = trace.posterior['phi'].values.flatten()

rng = np.random.default_rng(42)
n_draws = 500
ppc_samples = np.zeros((n_draws, N))

for i in range(n_draws):
    lp = psi_int_post[i] + X @ psi_coeff_post[i]
    pi_i = 1 / (1 + np.exp(-lp))
    lt = delta_int_post[i] + X @ delta_coeff_post[i]
    theta_i = 1 / (1 + np.exp(-lt))
    a_i, b_i = theta_i * phi_post[i], (1 - theta_i) * phi_post[i]
    u_i = rng.uniform(size=N)
    ppc_samples[i] = np.where(u_i < pi_i, 1.0, rng.beta(a_i, b_i))

# Two-panel posterior predictive check
obs_full_frac = (repayment == 1.0).mean()
obs_partial = repayment[repayment < 1.0]
ppc_full_fracs = (ppc_samples == 1.0).mean(axis=1)
ppc_partial = ppc_samples[ppc_samples < 1.0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4),
                                gridspec_kw={'width_ratios': [1, 2.5]})

# Left: spike proportion
ax1.bar(['Observed', 'Predicted'], [obs_full_frac, ppc_full_fracs.mean()],
        color=['steelblue', 'coral'], alpha=0.7, edgecolor='white')
ax1.errorbar(1, ppc_full_fracs.mean(),
             yerr=[[ppc_full_fracs.mean() - np.percentile(ppc_full_fracs, 3)],
                   [np.percentile(ppc_full_fracs, 97) - ppc_full_fracs.mean()]],
             fmt='none', color='black', capsize=5)
ax1.set_ylabel('Proportion', fontsize=12)
ax1.set_title('Fully Repaid (y = 1)', fontsize=12)
ax1.set_ylim(0, 0.75)

# Right: continuous component
ax2.hist(obs_partial, bins=40, density=True, alpha=0.5, color='steelblue',
         edgecolor='white', label='Observed')
ax2.hist(ppc_partial, bins=40, density=True, alpha=0.4, color='coral',
         edgecolor='white', label='Posterior predictive')
ax2.set_xlabel('Repayment Fraction', fontsize=12)
ax2.set_ylabel('Density', fontsize=12)
ax2.set_title('Partial Repayments (0 < y < 1)', fontsize=12)
ax2.legend(fontsize=10)
ax2.set_xlim(-0.02, 1.02)

plt.suptitle('Posterior Predictive Check', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 5. Expected Repayment by Covariate

The expected repayment combines both components: E[Y] = π + (1 - π) · θ.

Let's see how expected repayment varies with credit score while holding other covariates at their mean (0).

In [ ]:
credit_grid = np.linspace(-2, 2, 100)
X_grid = np.column_stack([credit_grid, np.zeros(100), np.zeros(100), np.zeros(100)])

n_post_samples = min(500, len(psi_int_post))
E_f_grid = np.zeros((n_post_samples, 100))
pi_grid = np.zeros((n_post_samples, 100))
theta_grid = np.zeros((n_post_samples, 100))

for i in range(n_post_samples):
    lp = psi_int_post[i] + X_grid @ psi_coeff_post[i]
    pi_i = 1 / (1 + np.exp(-lp))
    lt = delta_int_post[i] + X_grid @ delta_coeff_post[i]
    theta_i = 1 / (1 + np.exp(-lt))
    pi_grid[i] = pi_i
    theta_grid[i] = theta_i
    E_f_grid[i] = pi_i + (1 - pi_i) * theta_i

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, data, title, color in [
    (axes[0], pi_grid, 'P(Fully Repaid) = \u03C0', '#2196F3'),
    (axes[1], theta_grid, 'Partial Mean = \u03B8', '#FF9800'),
    (axes[2], E_f_grid, 'E[Repayment] = \u03C0 + (1-\u03C0)\u03B8', '#4CAF50'),
]:
    mean = data.mean(axis=0)
    lower = np.percentile(data, 3, axis=0)
    upper = np.percentile(data, 97, axis=0)
    ax.plot(credit_grid, mean, color=color, lw=2)
    ax.fill_between(credit_grid, lower, upper, color=color, alpha=0.15)
    ax.set_xlabel('Credit Score (standardised)', fontsize=11)
    ax.set_title(title, fontsize=12)
    ax.set_ylim(0, 1.02)

axes[0].set_ylabel('Probability / Fraction', fontsize=11)
plt.suptitle('Effect of Credit Score on Repayment Components', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Posterior Predictive Animation

Watch the posterior predictive distribution build up as we add more MCMC draws.

In [ ]:
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

draws_per_frame = [1, 2, 5, 10, 25, 50, 100, 200, 300, 400, 500]

fig_anim, (ax_s, ax_c) = plt.subplots(1, 2, figsize=(12, 4),
                                        gridspec_kw={'width_ratios': [1, 2.5]})

def update(frame_idx):
    ax_s.clear()
    ax_c.clear()
    n_draws = draws_per_frame[frame_idx]
    subset = ppc_samples[:n_draws]
    pred_full_frac = (subset == 1.0).mean()
    pred_partial = subset[subset < 1.0]

    ax_s.bar(['Observed', 'Predicted'], [obs_full_frac, pred_full_frac],
             color=['steelblue', 'coral'], alpha=0.7, edgecolor='white')
    ax_s.set_ylabel('Proportion')
    ax_s.set_title('Fully Repaid')
    ax_s.set_ylim(0, 0.75)

    ax_c.hist(obs_partial, bins=35, density=True, alpha=0.4, color='steelblue',
             edgecolor='white', label='Observed')
    if len(pred_partial) > 0:
        ax_c.hist(pred_partial, bins=35, density=True, alpha=0.4, color='coral',
                 edgecolor='white', label=f'Predicted ({n_draws} draws)')
    ax_c.set_xlabel('Repayment Fraction')
    ax_c.set_ylabel('Density')
    ax_c.set_title('Partial Repayments')
    ax_c.set_xlim(-0.02, 1.02)
    ax_c.set_ylim(0, 2.5)
    ax_c.legend(fontsize=9, loc='upper right')
    plt.suptitle(f'OIB Posterior Predictive ({n_draws} draws)', fontsize=13, y=1.02)
    plt.tight_layout()

anim = FuncAnimation(fig_anim, update, frames=len(draws_per_frame), interval=600)
plt.close(fig_anim)
HTML(anim.to_jshtml())

## Exercises

1. **Zero-Inflated Beta**: Modify the model to handle data with a spike at 0 instead of 1 (e.g., insurance claims where most customers file nothing). What changes in the log-likelihood?

2. **Heteroscedastic precision**: Give phi its own linear predictor (like we did for theta), so the Beta spread varies across borrowers. Does this improve the posterior predictive check?

3. **ZOIB model**: Extend to a Zero-and-One Inflated Beta for data with spikes at both boundaries. You will need three sub-models: P(y=0), P(y=1|y>0), and Beta for 0 < y < 1.

4. **Real data**: Find a real dataset with bounded outcomes and boundary inflation (e.g., student test completion rates, insurance loss ratios). Fit the OIB model and interpret the coefficients.

5. **CustomDist approach**: Rewrite the model using `pm.CustomDist` with a `logp` function that uses `pt.switch`. You will need to clip values to avoid Beta density evaluation at boundaries. Compare the sampling efficiency with the `pm.Potential` approach.